# Hiding a Payload in Model Weights (EvilModel demo)

**Team 8 — Part A (embedding).** This notebook actually *hides* a byte payload inside a small
network's weights and shows the model still classifies correctly. It adapts the **direct-substitution**
method (one payload byte per weight) from the team `evilmodel` implementation, which follows
EvilModel ([arXiv:2107.08590](https://arxiv.org/abs/2107.08590)). The model class and MNIST pipeline
are reused from the Module 6 *Information Hiding* example by **Angelica Liguori**.

Payloads here are **harmless** (a text string and random bytes) — no malware, per the project rules.

In [1]:
import os, struct, hashlib, copy, random
import numpy as np
import torch
import torch.nn as nn
import torchvision.datasets as datasets
import torchvision.transforms as transforms

seed = 101
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Target model

Set `TARGET` below. `"fc"` builds and briefly trains a tiny fully-connected net (our own model).
`"pretrained"` instead loads the shipped, already-trained `net.pt` (the Module 6 CNN) for an instant demo.
Everything after this cell is identical for both.

In [2]:
# Two model classes: our small fully-connected net, and the Module 6 CNN (only for the net.pt fallback).
class NetFC(nn.Module):
    # 784 -> 32 -> 10
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28 * 28, 32)
        self.fc2 = nn.Linear(32, 10)

    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(self.flatten(x))))


class NetCNN(nn.Module):
    # Module 6 example network; matches the shipped net.pt state_dict.
    def __init__(self, n_channels=1, h=28, w=28, last=64, hidden=8, n_classes=10):
        super().__init__()
        self.embedder = nn.Sequential(
            nn.Conv2d(n_channels, last // 2, 3, stride=2), nn.ReLU(),
            nn.Conv2d(last // 2, last, 3, stride=2), nn.ReLU(),
            nn.Conv2d(last, last, 3, stride=2), nn.ReLU(),
        )
        size = self.embedder(torch.rand(1, n_channels, h, w)).shape[2]
        self.fc1 = nn.Linear(last * size * size, last)
        self.fc2 = nn.Linear(last, hidden)
        self.out = nn.Linear(hidden, n_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.embedder(x).flatten(1)
        x = self.fc2(self.relu(self.fc1(x)))
        return self.out(self.relu(x))


transform = transforms.Compose([transforms.ToTensor()])
train_set = datasets.MNIST("./data", train=True, download=True, transform=transform)
test_set = datasets.MNIST("./data", train=False, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=256)


def accuracy(net):
    net.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            correct += (net(x).argmax(1) == y).sum().item()
            total += y.size(0)
    return 100 * correct / total


def quick_train(net, epochs=3):
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    net.train()
    for e in range(epochs):
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss_fn(net(x), y).backward()
            opt.step()
        print(f"  epoch {e + 1}/{epochs} done")

In [3]:
TARGET = "fc"   # "fc" = our small net; "pretrained" = load net.pt (Module 6 CNN)
FC_PATH = "pretrained_fc.pt"   # trained FC weights, saved once and reloaded for the demo

if TARGET == "fc":
    model = NetFC().to(device)
    if os.path.exists(FC_PATH):
        model.load_state_dict(torch.load(FC_PATH, weights_only=True, map_location=device))
        print(f"Loaded trained FC model from {FC_PATH}")
    else:
        print("Training the small fully-connected net...")
        quick_train(model, epochs=3)
        torch.save(model.state_dict(), FC_PATH)
        print(f"Saved trained FC model to {FC_PATH}")
elif TARGET == "pretrained":
    model = NetCNN().to(device)
    model.load_state_dict(torch.load("net.pt", weights_only=True, map_location=device))
else:
    raise ValueError("TARGET must be 'fc' or 'pretrained'")

clean_sd = copy.deepcopy(model.state_dict())   # keep an untouched copy
capacity = sum(p.numel() for p in clean_sd.values())
print(f"Model: {TARGET} | parameters (= byte capacity): {capacity:,}")
print(f"Baseline accuracy: {accuracy(model):.2f}%")

Loaded trained FC model from pretrained_fc.pt
Model: fc | parameters (= byte capacity): 25,450
Baseline accuracy: 94.60%


## 2. Embedding method

Each weight is a 32-bit float. We pack it **big-endian** and overwrite its **lowest `n_bytes`** with payload
bytes. The lowest byte holds the least-significant mantissa bits, so at `n_bytes=1` the weight changes by at
most ~2⁻¹⁵ of itself — far too little to affect accuracy, *no matter how many weights we fill*. Writing
**deeper** (`n_bytes=2,3`) raises capacity but reaches more significant bits and starts to hurt accuracy —
the capacity-vs-fidelity trade-off we show in §4. End of payload is marked with an escape sequence
(`deadbeef`); extraction reads those same low bytes back until it sees the marker.

Caveat: the payload must not contain the escape sequence (controlled here).

In [4]:
ESCAPE = b"\xde\xad\xbe\xef"


def embed_payload(sd, payload, n_bytes=1, escape=ESCAPE):
    # Writes payload+escape into the LOWEST n_bytes of successive weights (state_dict order).
    # n_bytes=1 is stealthy (no accuracy loss); larger n_bytes raises capacity but hurts accuracy.
    # Returns n_slots = number of weights used (the first n_slots are the embedded ones --
    # the hook a later 'freeze & retrain' step would use).
    data = payload + escape
    capacity = sum(p.numel() for p in sd.values()) * n_bytes
    if len(data) > capacity:
        raise ValueError(f"payload {len(data)} B exceeds capacity {capacity} B (n_bytes={n_bytes})")
    k = 0
    n_slots = 0
    with torch.no_grad():
        for p in sd.values():
            flat = p.view(-1)
            for i in range(flat.numel()):
                if k >= len(data):
                    return n_slots
                chunk = data[k:k + n_bytes]
                k += len(chunk)
                n_slots += 1
                packed = bytearray(struct.pack("!f", flat[i].item()))
                packed[4 - len(chunk):] = chunk          # overwrite the lowest len(chunk) bytes
                flat[i] = struct.unpack("!f", bytes(packed))[0]
    return n_slots


def extract_payload(sd, n_bytes=1, escape=ESCAPE):
    out = bytearray()
    for p in sd.values():
        for v in p.view(-1):
            out.extend(struct.pack("!f", v.item())[4 - n_bytes:])
            if out[-len(escape):] == escape:
                return bytes(out[:-len(escape)])
    return bytes(out)   # escape never found

## 3. Hide the payload and check the model still works

In [5]:
# --- Text payload: embed, re-check accuracy, read it back ---
text = b"Team 8 FoC demo: this stands in for a malware payload. Harmless text only."

sd_text = copy.deepcopy(clean_sd)
n = embed_payload(sd_text, text)
model.load_state_dict(sd_text)
acc_after = accuracy(model)
recovered = extract_payload(sd_text)

print(f"Text payload: {len(text)} B into {n} weights ({100 * n / capacity:.2f}% of capacity)")
print(f"Accuracy after embedding: {acc_after:.2f}%")
print(f"Recovered text: {recovered.decode()}")
print(f"Text round-trip OK: {recovered == text}")

model.load_state_dict(clean_sd)   # restore the clean model

# --- Random payload: embed, read back, verify by hash ---
blob = random.randbytes(1024)
sd_blob = copy.deepcopy(clean_sd)
embed_payload(sd_blob, blob)
recovered_blob = extract_payload(sd_blob)

print()
print(f"Random payload: {len(blob)} B")
print(f"sha256 original : {hashlib.sha256(blob).hexdigest()}")
print(f"sha256 recovered: {hashlib.sha256(recovered_blob).hexdigest()}")
assert recovered_blob == blob, "random-bytes round-trip failed"
print("Random round-trip OK: True")

Text payload: 74 B into 78 weights (0.31% of capacity)
Accuracy after embedding: 94.60%
Recovered text: Team 8 FoC demo: this stands in for a malware payload. Harmless text only.
Text round-trip OK: True

Random payload: 1024 B
sha256 original : 4971a9aaf10bc5cbd4c3ac0eccf3579629381a848e8c668cdf47ad53bdf8ada8
sha256 recovered: 4971a9aaf10bc5cbd4c3ac0eccf3579629381a848e8c668cdf47ad53bdf8ada8
Random round-trip OK: True


## 4. Pushing capacity: the fidelity trade-off

At `n_bytes=1` the payload size has **no** effect on accuracy, even when we fill every weight — that is the
whole point of LSB-style hiding. To make accuracy actually move we must write *deeper* into each weight.
Below we raise `n_bytes` to 3 (tripling capacity) and fill it, so higher mantissa/exponent bits get
overwritten and accuracy drops — while the payload still extracts back perfectly.

In [6]:
for n_bytes in (1, 2, 3):
    cap_nb = capacity * n_bytes
    big = random.randbytes(cap_nb - len(ESCAPE))          # fill the whole capacity at this depth
    sd_big = copy.deepcopy(clean_sd)
    embed_payload(sd_big, big, n_bytes=n_bytes)
    model.load_state_dict(sd_big)
    recovered_big = extract_payload(sd_big, n_bytes=n_bytes)
    print(f"n_bytes={n_bytes}: payload {len(big):6,} B (~{cap_nb/1024:.1f} KB) | "
          f"accuracy {accuracy(model):.2f}% | round-trip OK {recovered_big == big}")

model.load_state_dict(clean_sd)   # restore the clean model

n_bytes=1: payload 25,446 B (~24.9 KB) | accuracy 94.60% | round-trip OK True
n_bytes=2: payload 50,896 B (~49.7 KB) | accuracy 94.59% | round-trip OK True
n_bytes=3: payload 76,346 B (~74.6 KB) | accuracy 88.04% | round-trip OK True


<All keys matched successfully>

## 5. Optional extension: freeze & retrain (not implemented)

`embed_payload` returns `n_slots`, and the write order is deterministic, so the embedded weights are
exactly the **first `n_slots`** values in the `state_dict` traversal. A later robustness step could build
a mask over those positions, **freeze them** (restore their values or zero their gradient after each step),
and fine-tune the rest of the network to recover any lost accuracy — this is the *resilience training* /
robust-embedding idea. The embedding code above does not need to change to support it.

## How to run

1. `cd CyberSec-class-project && source .venv/bin/activate` (dependencies are pinned in `requirements.txt`).
2. Open this notebook and **Run All** from top to bottom.
3. Default `TARGET = "fc"` trains a tiny net (~30 s) the **first** time and saves it to `pretrained_fc.pt`;
   every later run (including the demo) just reloads that file instantly. Delete `pretrained_fc.pt` to
   retrain. Set `TARGET = "pretrained"` instead to use the Module 6 `net.pt`.

Expected: a baseline accuracy prints; after embedding, accuracy is essentially unchanged; the recovered
text prints identically; and the random-bytes sha256 check passes. `net.pt` is never modified.